In [1]:
import os
import warnings
import torch
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.sparse as sp
from process import utils
from process.utils import normalize_sparse_matrix, sparse_mx_to_torch_sparse_tensor
import scanpy as sc
plt.rcdefaults()
warnings.filterwarnings('ignore')
os.environ['R_HOME'] = r'E:/R/R-4.4.1'

In [2]:
sample = '151509'
path = 'F:/Code/spatial_domain/SGFST/SGFST/data/DLPFC'
adata = sc.read_visium(path=os.path.join(path, sample), count_file='filtered_feature_bc_matrix.h5', load_images=True)
adata.var_names_make_unique()
labels_df = pd.read_table(os.path.join(path, sample, "metadata.tsv"), sep='\t')
labels_df.index = adata.obs.index
adata.obs['ground_truth'] = labels_df["layer_guess_reordered"]
adata = adata[~adata.obs['ground_truth'].isnull()].copy()

In [3]:
sc.pp.filter_genes(adata, min_cells=3)
sc.pp.highly_variable_genes(adata, flavor='seurat_v3', n_top_genes=3000)
adata = adata[:, adata.var.highly_variable].copy()
adata.layers['counts'] = adata.X.copy()

In [4]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.scale(adata, zero_center=False, max_value=10)

In [5]:
signal_activity = utils.get_signal(adata, prior_file='F:/Code/spatial_domain/SGFST/SGFST/data/kegg/gene_sets.gmt', path=os.path.join(path), threads=1)
adata.obsm['signal'] = signal_activity.loc[adata.obs_names].values

spatial_adj, graph_nei = utils.spatial_construct_graph(adata, radius=300) # 560
signal_adj = utils.features_construct_graph(adata.obsm['signal'])

build expr_df time: 0.02s
write expr tsv time: 9.17s
R GSVA time: 256.51s
read output time: 0.21s
The graph contains 41788 undirected edges, 4788 cells.
17.4553 neighbors per cell on average.


In [6]:
features = torch.FloatTensor(adata.X.toarray())
labels = adata.obs['ground_truth'].values

spatial_adj = normalize_sparse_matrix(spatial_adj + sp.eye(adata.shape[0]))
spatial_adj = sparse_mx_to_torch_sparse_tensor(spatial_adj)

signal_adj = normalize_sparse_matrix(signal_adj + sp.eye(adata.shape[0]))
signal_adj = sparse_mx_to_torch_sparse_tensor(signal_adj)

graph_nei_tensor = torch.LongTensor(graph_nei.numpy())

In [7]:
def sym_nonneg_zero_diag(A):
    """
    对称 + 非负 + 去对角
    """
    A = 0.5 * (A + A.t())
    A = torch.clamp(A, min=0.0)
    A = A - torch.diag_embed(torch.diag(A))
    return A


def normalize_prob_graph(A, eps=1e-10):
    """
    概率图:
    P_ij = A_ij / vol(A)
    """
    A = sym_nonneg_zero_diag(A)
    vol = A.sum()
    if vol <= eps:
        n = A.size(0)
        P = torch.ones_like(A) / (n * n)
        P = P - torch.diag_embed(torch.diag(P))
        P = P / (P.sum() + eps)
        return P
    return A / (vol + eps)


def dsi_loss(A, B, eps=1e-10):
    """
    DSI(A, B) = | H(M) - 0.5*(H(A)+H(B)) |
    M = 0.5*(A/vol(A) + B/vol(B))
    """
    PA = normalize_prob_graph(A, eps=eps)
    PB = normalize_prob_graph(B, eps=eps)
    M = 0.5 * (PA + PB)

    HM = -(M * torch.log(M + eps)).sum()
    HA = -(PA * torch.log(PA + eps)).sum()
    HB = -(PB * torch.log(PB + eps)).sum()

    return torch.abs(HM - 0.5 * (HA + HB))


import torch
import torch.nn.functional as F


def huber_recon_loss(x_pred, x_true, delta=1.0):
    """
    Huber / SmoothL1 重构损失
    x_pred: decoder 输出, shape [n, p]
    x_true: 输入特征, shape [n, p]
    """
    return F.smooth_l1_loss(x_pred, x_true, beta=delta, reduction='mean')


import torch
import torch.nn.functional as F

import torch
import torch.nn.functional as F


def neighborhood_infonce_loss_fast(embd, graph_nei, temperature=0.5, eps=1e-8):
    """
    更快的全矩阵版 InfoNCE
    embd: [n, d]
    graph_nei: [n, n]，0/1邻接矩阵
    """
    device = embd.device
    n = embd.size(0)

    # 归一化
    z = F.normalize(embd, p=2, dim=1)

    # 相似度矩阵
    sim = torch.matmul(z, z.t()) / temperature  # [n, n]

    # 去掉对角线
    diag_mask = torch.eye(n, dtype=torch.bool, device=device)

    pos_mask = graph_nei.bool() & (~diag_mask)  # 正样本
    valid_mask = ~diag_mask  # 所有可参与分母的样本

    # 数值稳定：每行减去最大值
    sim = sim - sim.max(dim=1, keepdim=True)[0].detach()

    exp_sim = torch.exp(sim) * valid_mask.float()

    # 分子：正样本和
    pos_sum = (exp_sim * pos_mask.float()).sum(dim=1)  # [n]

    # 分母：除自己外所有样本
    all_sum = exp_sim.sum(dim=1) + eps  # [n]

    # 只保留有正邻居的点
    valid_nodes = pos_mask.sum(dim=1) > 0
    if valid_nodes.sum() == 0:
        return torch.tensor(0.0, device=device, requires_grad=True)

    loss = -torch.log((pos_sum[valid_nodes] + eps) / all_sum[valid_nodes])
    return loss.mean()

In [8]:
import torch
import torch.nn.functional as F


def bpr_structure_loss(scores, adj, num_neg=3, num_pos=4096):
    """
    scores: [n, n]，模型输出的 rec_adj（不要 sigmoid）
    adj:    [n, n]，0/1邻接矩阵 graph_nei
    num_neg: 每个正边采几个负样本
    num_pos: 每轮随机采多少条正边；越大越准，越小越快
    """
    device = scores.device
    n = adj.size(0)

    # bool邻接，去掉对角线
    adj_bool = (adj > 0)
    diag_mask = torch.eye(n, dtype=torch.bool, device=device)
    adj_bool = adj_bool & (~diag_mask)

    # 无向图：只取上三角正边，避免重复
    pos_idx = torch.triu(adj_bool, diagonal=1).nonzero(as_tuple=False)

    if pos_idx.size(0) == 0:
        return scores.sum() * 0.0

    # 随机采样部分正边，加速
    if (num_pos is not None) and (pos_idx.size(0) > num_pos):
        perm = torch.randperm(pos_idx.size(0), device=device)[:num_pos]
        pos_idx = pos_idx[perm]

    src = pos_idx[:, 0]                     # [m]
    pos = pos_idx[:, 1]                     # [m]
    m = src.size(0)

    # 随机采负样本
    neg = torch.randint(0, n, (m, num_neg), device=device)   # [m, num_neg]
    src_ex = src.unsqueeze(1)                                # [m, 1]

    # 不能采到自己，也不能采到真实邻居
    invalid = (neg == src_ex) | adj_bool[src_ex, neg]

    # 少量重采样，基本够用
    for _ in range(5):
        if not invalid.any():
            break
        neg[invalid] = torch.randint(0, n, (invalid.sum().item(),), device=device)
        invalid = (neg == src_ex) | adj_bool[src_ex, neg]

    # BPR: 希望 pos_score > neg_score
    pos_score = scores[src, pos].unsqueeze(1)    # [m, 1]
    neg_score = scores[src_ex, neg]              # [m, num_neg]

    loss = F.softplus(-(pos_score - neg_score)).mean()
    return loss

In [9]:
import random
import torch.nn as nn

def model_train(model, optimizer, features, spatial_adj, signal_adj, graph_nei, spatial_adj_dsi, signal_adj_dsi, alpha=1, beta=0.1,gama=0.05):
# def model_train(model, optimizer, features, spatial_adj, signal_adj, graph_nei, alpha=1, beta=0.1):
    np.random.seed(42)
    random.seed(42)
    torch.manual_seed(42)
    torch.cuda.manual_seed(42)

    model.train()
    optimizer.zero_grad()
    # embd, rec_adj, shared_graph,pi, disp, mean, att = model(features, spatial_adj, signal_adj)
    embd, rec_adj, pi, disp, mean, att,shared_graph = model(features, spatial_adj, signal_adj)

    structure_loss = bpr_structure_loss(rec_adj, graph_nei, num_neg=3, num_pos=4096)
    # huber_loss = huber_recon_loss(mean, features, delta=1.0)
    # infonce_loss = neighborhood_infonce_loss_fast(
    #     embd, graph_nei, temperature=0.5    )
    # dsi loss
    shared_graph = sym_nonneg_zero_diag(shared_graph)
    spatial_graph = sym_nonneg_zero_diag(spatial_adj_dsi)
    signal_graph = sym_nonneg_zero_diag(signal_adj_dsi)

    dsi_spatial = dsi_loss(shared_graph, spatial_graph)
    dsi_signal = dsi_loss(shared_graph, signal_graph)
    consistency_loss = dsi_spatial + dsi_signal
    
    zinb_loss = SGFST.ZINB(pi, theta=disp, ridge_lambda=0).loss(features, mean, mean=True)
# 建议在 optimizer.step() 后添加
    total_loss = alpha * zinb_loss + beta * structure_loss+gama*consistency_loss
   
    total_loss.backward()
    optimizer.step()

    return embd, att, total_loss.item(), pi, disp, mean, att, shared_graph
    # return embd, att, total_loss.item(), pi, disp, mean, att

In [10]:
spatial_adj_raw, graph_nei = utils.spatial_construct_graph(adata, radius=300)
signal_adj_raw = utils.features_construct_graph(adata.obsm['signal'])
# 给 DSI 用的 dense 图
spatial_adj_dsi = torch.FloatTensor(spatial_adj_raw.toarray())

if sp.issparse(signal_adj_raw):
    signal_adj_dsi = torch.FloatTensor(signal_adj_raw.toarray())
else:
    signal_adj_dsi = torch.FloatTensor(np.asarray(signal_adj_raw))

The graph contains 41788 undirected edges, 4788 cells.
17.4553 neighbors per cell on average.


In [11]:
from tqdm import tqdm
from model import SGFST
from sklearn.metrics import adjusted_rand_score
from sklearn.cluster import KMeans

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

features, spatial_adj, signal_adj = features.to(device), spatial_adj.to(device), signal_adj.to(device)
graph_nei = graph_nei.to(device)

model = SGFST.SGFST(nfeat=features.shape[1], nhid1=128, nhid2=64, dropout=0.1).to(device)
optimizer = torch.optim.NAdam(model.parameters(), lr=0.001, weight_decay=5e-4)

ari_max = 0
best_emd = None
best_cluster = None
best_mean = None

for epoch in tqdm(range(1, 501), leave=True, desc="Training epochs"):
    model.train()
    optimizer.zero_grad()
    embd, recon_adj,loss, pi, disp, mean, att, shared_graph = model_train(model, optimizer, features, spatial_adj, signal_adj, graph_nei,spatial_adj_dsi,signal_adj_dsi, alpha=1, beta=0.1,gama=0.01)
    # embd, recon_adj, loss, pi, disp, mean, att = model_train(model, optimizer, features, spatial_adj, signal_adj, graph_nei, alpha=1, beta=0.1)

    kmeans = KMeans(n_clusters=len(np.unique(labels))).fit(embd.cpu().detach().numpy())
    ari_res = adjusted_rand_score(labels, kmeans.labels_)
    if (epoch % 25 == 0) or (epoch ==1):
        print(f"Epoch: {epoch:03d}, Loss: {loss:.4f}, ARI: {ari_res:.4f}")

    if ari_res > ari_max:
        ari_max = ari_res
        best_emb = embd
        best_clusters = kmeans.labels_
        best_mean = mean

print(f"\n--- Training complete. ARI: {ari_max:.4f} ---")
#1,0.1,0.05===0.6710; 1,0.08,0.01===0.6784

Training epochs:   0%|          | 1/500 [00:05<42:03,  5.06s/it]

Epoch: 001, Loss: 0.6011, ARI: 0.1232


Training epochs:   5%|▌         | 25/500 [01:54<36:14,  4.58s/it]

Epoch: 025, Loss: 0.4892, ARI: 0.4514


Training epochs:  10%|█         | 50/500 [03:46<33:34,  4.48s/it]

...

Epoch: 450, Loss: 0.4113, ARI: 0.3816


Training epochs:  95%|█████████▌| 475/500 [36:03<01:50,  4.41s/it]

Epoch: 475, Loss: 0.4116, ARI: 0.3551


Training epochs: 100%|██████████| 500/500 [37:57<00:00,  4.55s/it]

Epoch: 500, Loss: 0.4133, ARI: 0.3524

--- Training complete. ARI: 0.5849 ---


In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment

# 1. 设置聚类结果和真实标签
adata.obs['domain'] = pd.Categorical(best_clusters)
adata.obs['ground_truth'] = pd.Categorical(adata.obs['ground_truth'])

# 2. 构造列联表：domain × ground_truth
ct = pd.crosstab(adata.obs['domain'], adata.obs['ground_truth'])

# 3. 匈牙利算法做最佳一一匹配
row_ind, col_ind = linear_sum_assignment(-ct.values)
mapping = {ct.index[r]: ct.columns[c] for r, c in zip(row_ind, col_ind)}

print("domain -> ground_truth 对应关系:")
print(mapping)

# 4. 先准备 ground_truth 的类别顺序
gt_cats = list(adata.obs['ground_truth'].cat.categories)

# 5. 给 ground_truth 准备统一颜色
if 'ground_truth_colors' in adata.uns and len(adata.uns['ground_truth_colors']) == len(gt_cats):
    gt_colors = list(adata.uns['ground_truth_colors'])
else:
    gt_colors = [   "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf" ][:len(gt_cats)]
adata.uns['ground_truth_colors'] = gt_colors

# 6. 建立 ground_truth 类别 -> 颜色 的字典
gt_color_dict = dict(zip(gt_cats, gt_colors))

# 7. 按照 mapping，给 domain 的每个类别分配颜色
domain_cats = list(adata.obs['domain'].cat.categories)
domain_colors = []
for d in domain_cats:
    matched_gt = mapping[d]
    domain_colors.append(gt_color_dict[matched_gt])

adata.uns['domain_colors'] = domain_colors
# 8. 画图
plt.rcParams["figure.figsize"] = (5, 5)
sc.pl.spatial(    adata,    img_key="hires",    color=['domain', 'ground_truth'],    title=[f'SGFST (ARI: {ari_max:.4f})', 'Ground Truth'],
    show=True,    size=1.5,)

In [13]:

import numpy as np
from scipy.spatial.distance import cdist
from collections import Counter

def refine_label(adata, radius=50, key='label'):
    old_type = np.asarray(adata.obs[key].values)
    position = np.asarray(adata.obsm['spatial'])

    # 两两距离
    distance = cdist(position, position, metric='euclidean')

    new_type = []
    for i in range(distance.shape[0]):
        # 从近到远排序，第0个是自己
        index = np.argsort(distance[i])[1:radius+1]
        neigh_type = old_type[index]

        # 多数投票
        max_type = Counter(neigh_type).most_common(1)[0][0]
        new_type.append(str(max_type))

    return new_type

In [14]:
from process.utils import BestMap

sc.set_figure_params(scanpy=True, dpi=80, dpi_save=600, frameon=True, vector_friendly=False, fontsize=12, figsize=(5, 4), color_map=None, format='pdf', facecolor=None, transparent=True, ipython_format='png2x')

new_type = refine_label(adata, 20, key='domain')
adata.obs["refined_pred"] = new_type
adata.obs["refined_pred"] = adata.obs["refined_pred"].astype('category')

adata.obs['pred'] = BestMap(pd.Categorical(adata.obs['ground_truth']).codes, pd.Categorical(adata.obs['domain']).codes)
adata.obs['pred'] = adata.obs['pred'].astype('category')

adata.obs['refined_pred'] = BestMap(pd.Categorical(adata.obs['ground_truth']).codes, pd.Categorical(adata.obs['refined_pred']).codes)
adata.obs['refined_pred'] = adata.obs['refined_pred'].astype('category')

In [15]:
ari_refine = adjusted_rand_score(adata.obs['ground_truth'], adata.obs['refined_pred'])
ari_refine

0.5842295424230226